In [1]:
import pandas as pd

# Load data
df = pd.read_csv("../data/processed/loan_risk_gold_full.csv")

# Basic info
print("="*50)
print("Dataset Shape")
print(df.shape)

print("\nColumns")
print(df.columns.tolist())

print("\nTarget Distribution")
print(df["target"].value_counts())

print("\nTarget Ratio")
print(df["target"].value_counts(normalize=True))

# Remove ID
X = df.drop(columns=["sk_id_curr", "target"])

# Target
y = df["target"]

print("\nFeature Shape")
print(X.shape)

print("\nTarget Shape")
print(y.shape)

# Detect column types
categorical_cols = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_cols = X.select_dtypes(
    exclude=["object", "category"]
).columns.tolist()

print("\nCategorical Columns:", len(categorical_cols))
print(categorical_cols)

print("\nNumeric Columns:", len(numeric_cols))
print(numeric_cols[:20])   # first 20

# Missing values
missing = X.isnull().sum()

missing = (
    missing[missing > 0]
    .sort_values(ascending=False)
)

print("\nTop Missing Columns")
print(missing.head(20))

Dataset Shape
(307511, 136)

Columns
['sk_id_curr', 'target', 'name_contract_type', 'code_gender', 'flag_own_car', 'flag_own_realty', 'cnt_children', 'amt_income_total', 'amt_credit', 'amt_annuity', 'amt_goods_price', 'name_type_suite', 'name_income_type', 'name_education_type', 'name_family_status', 'name_housing_type', 'region_population_relative', 'days_birth', 'days_employed', 'days_registration', 'days_id_publish', 'own_car_age', 'flag_mobil', 'flag_emp_phone', 'flag_work_phone', 'flag_cont_mobile', 'flag_phone', 'flag_email', 'occupation_type', 'cnt_fam_members', 'region_rating_client', 'region_rating_client_w_city', 'weekday_appr_process_start', 'hour_appr_process_start', 'reg_region_not_live_region', 'reg_region_not_work_region', 'live_region_not_work_region', 'reg_city_not_live_city', 'reg_city_not_work_city', 'live_city_not_work_city', 'organization_type', 'ext_source_1', 'ext_source_2', 'ext_source_3', 'apartments_avg', 'basementarea_avg', 'years_beginexpluatation_avg', 'yea

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_24524\2333514932.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(


In [2]:
from sklearn.model_selection import train_test_split

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\ny_train distribution")
print(y_train.value_counts(normalize=True))

print("\ny_test distribution")
print(y_test.value_counts(normalize=True))


# Calculate class imbalance weight

n_negative = (y_train == 0).sum()
n_positive = (y_train == 1).sum()

scale_pos_weight = n_negative / n_positive

print("\nNegative samples:", n_negative)
print("Positive samples:", n_positive)

print("\nscale_pos_weight =", round(scale_pos_weight, 2))

X_train: (246008, 134)
X_test : (61503, 134)

y_train distribution
target
0    0.919271
1    0.080729
Name: proportion, dtype: float64

y_test distribution
target
0    0.919272
1    0.080728
Name: proportion, dtype: float64

Negative samples: 226148
Positive samples: 19860

scale_pos_weight = 11.39


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import OneHotEncoder

import lightgbm as lgb


# -------------------------
# Numeric preprocessing
# -------------------------

numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]
)


# -------------------------
# Categorical preprocessing
# -------------------------

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),

        (
            "encoder",

            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)


# -------------------------
# Combine preprocessors
# -------------------------

preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",

            numeric_transformer,

            numeric_cols
        ),

        (
            "cat",

            categorical_transformer,

            categorical_cols
        )

    ]
)


# -------------------------
# LightGBM model
# -------------------------

model = lgb.LGBMClassifier(

    objective="binary",

    metric="auc",

    n_estimators=500,

    learning_rate=0.05,

    num_leaves=63,

    scale_pos_weight=scale_pos_weight,

    random_state=42,

    verbose=-1
)


# -------------------------
# Full Pipeline
# -------------------------

pipeline = Pipeline(

    steps=[

        (
            "preprocessor",

            preprocessor
        ),

        (
            "model",

            model
        )
    ]
)


print(pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['cnt_children',
                                                   'amt_income_total',
                                                   'amt_credit', 'amt_annuity',
                                                   'amt_goods_price',
                                                   'region_population_relative',
                                                   'days_birth',
                                                   'days_employed',
                                                   'days_registration',
                                                   'days_id_publish',
                                                   'own_car_age', 'flag_mobil',
    

In [4]:
from time import time

start = time()

pipeline.fit(
    X_train,
    y_train
)

end = time()

print(f"Training Time: {end-start:.2f} seconds")

Training Time: 27.25 seconds


In [5]:
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Probability predictions

y_prob = pipeline.predict_proba(X_test)[:, 1]

# Binary predictions

y_pred = pipeline.predict(X_test)


# Metrics

auc = roc_auc_score(y_test, y_prob)

acc = accuracy_score(y_test, y_pred)

precision = precision_score(y_test, y_pred)

recall = recall_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)


print("="*50)

print("ROC AUC :", round(auc,4))

print("Accuracy :", round(acc,4))

print("Precision :", round(precision,4))

print("Recall :", round(recall,4))

print("F1 Score :", round(f1,4))

print("\nConfusion Matrix")

print(cm)

c:\Users\LENOVO\Downloads\Datasets\home-credit-default-risk\home-default-risk\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\LENOVO\Downloads\Datasets\home-credit-default-risk\home-default-risk\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


ROC AUC : 0.7583
Accuracy : 0.7625
Precision : 0.1918
Recall : 0.6042
F1 Score : 0.2912

Confusion Matrix
[[43899 12639]
 [ 1965  3000]]


In [6]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(
    pipeline,
    "../models/pipeline.pkl"
)

print("Pipeline saved successfully.")

Pipeline saved successfully.


In [7]:
loaded_pipeline = joblib.load(
    "../models/pipeline.pkl"
)

print(type(loaded_pipeline))

<class 'sklearn.pipeline.Pipeline'>


In [8]:
sample = X_test.iloc[[0]]

pred_prob = loaded_pipeline.predict_proba(sample)[0][1]

pred_class = loaded_pipeline.predict(sample)[0]

print("Probability:", pred_prob)

print("Prediction:", pred_class)

Probability: 0.6929441862417139
Prediction: 1


c:\Users\LENOVO\Downloads\Datasets\home-credit-default-risk\home-default-risk\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\LENOVO\Downloads\Datasets\home-credit-default-risk\home-default-risk\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [9]:
# Get feature names after preprocessing

feature_names = pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

# Get LightGBM model

model = pipeline.named_steps["model"]

# Feature importances

importances = model.feature_importances_

import pandas as pd

importance_df = pd.DataFrame({

    "feature": feature_names,

    "importance": importances

})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

importance_df.head(30)

,feature,importance
29,num__ext_source_3,1291
28,num__ext_source_2,1203
6,num__days_birth,1194
105,num__bureau_recent_credit,1092
9,num__days_id_publish,1035
8,num__days_registration,1017
77,num__days_last_phone_change,1011
27,num__ext_source_1,1009
3,num__amt_annuity,1005
104,num__bureau_avg_credit,985
